# Setup Cell
Run this first to load the dataset and prepare `X_train` and `y_train` as 2D arrays for the multi-label `OneVsRestClassifier`.

In [10]:
import sys
import os
import pandas as pd
from sklearn.model_selection import train_test_split

# Ensure we are in the project root to import src natively
if os.getcwd().endswith('notebooks'):
    os.chdir('..')
sys.path.append(os.getcwd())

from src.preprocess import clean_text

print("Loading dataset...")
data_path = r"..\data\archive (6)\train.csv"
if not os.path.exists(data_path):
    data_path = "data/archive (6)/train.csv"
df_full = pd.read_csv(data_path)

print("Sampling 20,000 rows (stratified)...")
df, _ = train_test_split(
    df_full,
    train_size=20000,
    stratify=df_full["toxic"],
    random_state=42
)

print("Cleaning text...")
df["comment_text"] = df["comment_text"].astype(str).apply(clean_text)

print("Splitting train/test...")
LABELS = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]

X_train, X_test, y_train, y_test = train_test_split(
    df["comment_text"].tolist(),
    df[LABELS].values,  # <--- ALL 6 labels (Multi-label 2D array)
    test_size=0.2,
    random_state=42,
    stratify=df["toxic"]
)

print(f"X_train samples: {len(X_train)}")
print(f"y_train shape:   {y_train.shape}")
print("Setup complete! Proceed to the next cell.")

Loading dataset...
Sampling 20,000 rows (stratified)...
Cleaning text...
Splitting train/test...
X_train samples: 16000
y_train shape:   (16000, 6)
Setup complete! Proceed to the next cell.


# Multi-Label Training & Tuning Cell
This is the exact code block originally provided using `OneVsRestClassifier`.

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report
import pandas as pd
import json

LABELS = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]

# Build pipeline
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=50000,
        ngram_range=(1, 2),
        sublinear_tf=True,
        min_df=3
    )),
    ("clf", OneVsRestClassifier(
        LogisticRegression(
            class_weight="balanced",   # handles 90/10 imbalance
            max_iter=1000,
            random_state=42
        )
    ))
])

# Hyperparameter grid to search over
param_grid = {
    "tfidf__max_features": [30000, 50000],
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "clf__estimator__C": [0.1, 1.0, 10.0],
}

# Run grid search with 3-fold cross-validation, optimising macro F1
print("Running GridSearchCV — this takes ~5 minutes...")
grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=3,
    scoring="f1_macro",
    n_jobs=-1,
    verbose=1
)
grid_search.fit(X_train, y_train)

print(f"\n✅ Best parameters found:")
for param, value in grid_search.best_params_.items():
    print(f"   {param}: {value}")
print(f"   Best CV F1 (macro): {grid_search.best_score_:.4f}")

# Evaluate best model on held-out test set
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

# Per-label metrics
print("\n" + "="*60)
print("BASELINE RESULTS — TF-IDF + Logistic Regression")
print("="*60)
for i, label in enumerate(LABELS):
    f1 = f1_score(y_test[:, i], y_pred[:, i], average="binary")
    prec = precision_score(y_test[:, i], y_pred[:, i], average="binary", zero_division=0)
    rec = recall_score(y_test[:, i], y_pred[:, i], average="binary", zero_division=0)
    print(f"  {label:<15} F1: {f1:.4f}  Precision: {prec:.4f}  Recall: {rec:.4f}")

# Overall weighted F1
f1_weighted = f1_score(y_test, y_pred, average="weighted")
f1_macro = f1_score(y_test, y_pred, average="macro")
print(f"\n  Weighted F1: {f1_weighted:.4f}")
print(f"  Macro F1:    {f1_macro:.4f}")
print("="*60)

# Save metrics to outputs/
metrics = {
    "model": "TF-IDF + Logistic Regression (tuned)",
    "best_params": grid_search.best_params_,
    "best_cv_f1_macro": round(grid_search.best_score_, 4),
    "f1_weighted": round(f1_weighted, 4),
    "f1_macro": round(f1_macro, 4),
    "per_label": {
        label: {
            "f1": round(f1_score(y_test[:, i], y_pred[:, i], average="binary"), 4),
            "precision": round(precision_score(y_test[:, i], y_pred[:, i], average="binary", zero_division=0), 4),
            "recall": round(recall_score(y_test[:, i], y_pred[:, i], average="binary", zero_division=0), 4)
        }
        for i, label in enumerate(LABELS)
    }
}

import os
os.makedirs("outputs", exist_ok=True)
with open("outputs/baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("\n✅ Metrics saved to outputs/baseline_metrics.json")
print("📋 Copy these into your paper.md results table!")

Running GridSearchCV — this takes ~5 minutes...
Fitting 3 folds for each of 12 candidates, totalling 36 fits

✅ Best parameters found:
   clf__estimator__C: 10.0
   tfidf__max_features: 30000
   tfidf__ngram_range: (1, 1)
   Best CV F1 (macro): 0.5686

BASELINE RESULTS — TF-IDF + Logistic Regression
  toxic           F1: 0.7244  Precision: 0.7282  Recall: 0.7206
  severe_toxic    F1: 0.4510  Precision: 0.3651  Recall: 0.5897
  obscene         F1: 0.7628  Precision: 0.7761  Recall: 0.7500
  threat          F1: 0.4706  Precision: 0.5714  Recall: 0.4000
  insult          F1: 0.6492  Precision: 0.6392  Recall: 0.6596
  identity_hate   F1: 0.3714  Precision: 0.3514  Recall: 0.3939

  Weighted F1: 0.6884
  Macro F1:    0.5716

✅ Metrics saved to outputs/baseline_metrics.json
📋 Copy these into your paper.md results table!
